# Part 1b — Llama Guard 2 row of Table 1

Computes the **Llama Guard 2** row of Table 1 (Judge Selection) using
`meta-llama/Meta-Llama-Guard-2-8B` on the 300-prompt JBB judge_comparison split.

Lives in its own kernel for the same load-order reason as the Guard-3 appendix
(Part 11) — Guard-2-8B is ~16 GB and races with any other vLLM-resident model on
a 24 GB GPU.

`10_summary.ipynb` joins Part 1 (rule-based), Part 1b (Guard-2), and Part 11
(Guard-3) into a single Table 1.

**Outputs**: `results_part1b/summary.csv`, `results_part1b/raw.json`.


In [ ]:
import sys
sys.path.insert(0, '.')

from jbb_common import (
    setup_vllm_env, patch_jbb, hf_login, load_jbb_dataset,
    judge_metrics, refusal_keyword_check,
    format_llama2_chat, format_llama3_chat,
    JBB_SYSTEM_PROMPT, REFUSAL_KEYWORDS,
)
setup_vllm_env()
patch_jbb()
hf_login()
goals, benign_goals, judge_df = load_jbb_dataset()
print(f"Loaded {len(goals)} harmful goals, {len(benign_goals)} benign goals, "
      f"{len(judge_df)} judge rows")


In [ ]:
import os, json, gc
import pandas as pd
import torch
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

judge_prompts   = judge_df["prompt"].tolist()
judge_responses = judge_df["target_response"].tolist()

human_majority = judge_df["human_majority"]
if human_majority.dtype == bool:
    ground_truth = human_majority.tolist()
elif human_majority.dtype == object:
    ground_truth = human_majority.str.lower().isin(
        ["unsafe", "yes", "jailbroken", "true"]
    ).tolist()
else:
    ground_truth = human_majority.tolist()
print(f"Ground truth: {sum(ground_truth)} unsafe / "
      f"{len(ground_truth) - sum(ground_truth)} safe")

LG2_MODEL = "meta-llama/Meta-Llama-Guard-2-8B"
lg2_tokenizer = AutoTokenizer.from_pretrained(LG2_MODEL)

# (prompt, response) -> 2-turn chat with add_generation_prompt=True so Guard-2
# emits its safe / unsafe verdict for the assistant turn (same template family
# as Guard-3).
lg2_prompts = [
    lg2_tokenizer.apply_chat_template(
        [{"role": "user",      "content": p},
         {"role": "assistant", "content": r}],
        tokenize=False, add_generation_prompt=True,
    )
    for p, r in zip(judge_prompts, judge_responses)
]

llm_lg2 = LLM(
    model=LG2_MODEL, dtype="float16",
    max_model_len=4096, gpu_memory_utilization=0.90,
    tensor_parallel_size=1, enforce_eager=True,
)
lg2_sp = SamplingParams(temperature=0.0, max_tokens=50)
lg2_outputs = llm_lg2.generate(lg2_prompts, lg2_sp)

lg2_preds = []
for output in lg2_outputs:
    text = output.outputs[0].text.strip().lower()
    first_line = text.split("\n")[0]
    lg2_preds.append(first_line == "unsafe")

del llm_lg2, lg2_tokenizer
gc.collect()
torch.cuda.empty_cache()

metrics_lg2 = judge_metrics(lg2_preds, ground_truth)
print("Llama Guard 2 metrics:")
for k, v in metrics_lg2.items():
    print(f"  {k}: {v}")


In [ ]:
os.makedirs("results_part1b", exist_ok=True)

summary_df = pd.DataFrame([{
    "Classifier": "Llama Guard 2",
    "Agreement":  metrics_lg2["agreement"],
    "FPR":        metrics_lg2["fpr"],
    "FNR":        metrics_lg2["fnr"],
}])
summary_df.to_csv("results_part1b/summary.csv", index=False)

with open("results_part1b/raw.json", "w") as f:
    json.dump({
        "predictions":  [bool(p) for p in lg2_preds],
        "ground_truth": [bool(g) for g in ground_truth],
        "metrics":      metrics_lg2,
    }, f, indent=2)

print("\n### Table 1 (Part 1b row): Llama Guard 2 judge ###")
print(summary_df.to_string(index=False))
